# VayuVision — Data Understanding

## Objective

This notebook performs an initial inspection of the India city-level air-quality dataset.

The purpose is to understand the dataset before cleaning it or building a machine-learning model.

## Dataset overview

Each row represents air-quality observations for one city on one date.

The dataset includes:

- City and date information
- Pollutant measurements, such as PM2.5, PM10, NO2, SO2, CO, and O3
- AQI, which represents overall air quality
- AQI bucket, which gives the AQI category in readable form

## Initial checks performed

In this notebook, I inspect:

- Dataset shape
- Column names and data types
- Cities included in the dataset
- Date coverage
- Missing values
- Duplicate rows
- Summary statistics
- AQI bucket categories

## Why this step matters

Understanding the raw data helps identify data-quality issues before analysis and modelling.

AQI will be used as the main outcome variable because it represents overall pollution levels. Pollutant columns and historical AQI values will later be used to predict next-day AQI.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("data/raw/city_day.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("../data/raw/city_day.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset path: {DATA_PATH}")
print(f"Dataset shape: {df.shape}")
df.sample(5)

Dataset path: ..\data\raw\city_day.csv
Dataset shape: (29531, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
19326,Kolkata,2019-01-29,97.05,175.72,33.82,83.43,113.39,23.96,0.75,11.30,29.48,0.59,0.90,NaN,233.0,Poor
7043,Brajrajnagar,2019-03-03,97.36,166.06,21.80,21.02,37.88,37.46,3.02,3.45,5.10,0.00,NaN,NaN,277.0,Poor
7754,Chandigarh,2020-04-13,19.73,44.85,1.68,10.72,6.95,31.33,0.54,9.92,26.65,3.43,0.63,1.16,45.0,Good
29280,Visakhapatnam,2019-10-25,37.82,88.58,22.91,56.87,48.88,1.16,1.44,22.24,19.29,5.48,12.05,3.52,121.0,Moderate
10174,Coimbatore,2020-05-08,26.23,30.39,0.99,44.79,45.75,NaN,1.42,6.48,18.95,0.00,0.04,NaN,74.0,Satisfactory


In [3]:
df.info()

print("\nColumn data types:")
display(df.dtypes.to_frame(name="Data type"))

<class 'pandas.DataFrame'>
RangeIndex: 29531 entries, 0 to 29530
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   City        29531 non-null  str    
 1   Date        29531 non-null  str    
 2   PM2.5       24933 non-null  float64
 3   PM10        18391 non-null  float64
 4   NO          25949 non-null  float64
 5   NO2         25946 non-null  float64
 6   NOx         25346 non-null  float64
 7   NH3         19203 non-null  float64
 8   CO          27472 non-null  float64
 9   SO2         25677 non-null  float64
 10  O3          25509 non-null  float64
 11  Benzene     23908 non-null  float64
 12  Toluene     21490 non-null  float64
 13  Xylene      11422 non-null  float64
 14  AQI         24850 non-null  float64
 15  AQI_Bucket  24850 non-null  str    
dtypes: float64(13), str(3)
memory usage: 4.3 MB

Column data types:


,Data type
City,str
Date,str
PM2.5,float64
PM10,float64
NO,float64
NO2,float64
NOx,float64
NH3,float64
CO,float64
SO2,float64


In [4]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print("Number of unique cities:", df["City"].nunique())
print("Cities:")
print(sorted(df["City"].dropna().unique()))

print("\nDate range:")
print("Start:", df["Date"].min().date())
print("End:", df["Date"].max().date())
print("Invalid/missing dates:", df["Date"].isna().sum())

Number of unique cities: 26
Cities:
['Ahmedabad', 'Aizawl', 'Amaravati', 'Amritsar', 'Bengaluru', 'Bhopal', 'Brajrajnagar', 'Chandigarh', 'Chennai', 'Coimbatore', 'Delhi', 'Ernakulam', 'Gurugram', 'Guwahati', 'Hyderabad', 'Jaipur', 'Jorapokhar', 'Kochi', 'Kolkata', 'Lucknow', 'Mumbai', 'Patna', 'Shillong', 'Talcher', 'Thiruvananthapuram', 'Visakhapatnam']

Date range:
Start: 2015-01-01
End: 2020-07-01
Invalid/missing dates: 0


In [5]:
missing_values = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame(name="Missing values")
)

missing_values["Missing percentage"] = (
    missing_values["Missing values"] / len(df) * 100
).round(2)

display(missing_values)

,Missing values,Missing percentage
Xylene,18109,61.32
PM10,11140,37.72
NH3,10328,34.97
Toluene,8041,27.23
Benzene,5623,19.04
AQI,4681,15.85
AQI_Bucket,4681,15.85
PM2.5,4598,15.57
NOx,4185,14.17
O3,4022,13.62


In [6]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_count / len(df) * 100, 2), "%")

Duplicate rows: 0
Duplicate percentage: 0.0 %


In [7]:
display(df.describe(include="all").T)

print("\nAQI bucket counts:")
display(df["AQI_Bucket"].value_counts(dropna=False).to_frame(name="Count"))

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
City,29531,26,Ahmedabad,2009,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,29531,NaN,NaN,NaN,2018-05-14 05:40:15.807118,2015-01-01 00:00:00,2017-04-16 00:00:00,2018-08-05 00:00:00,2019-09-03 00:00:00,2020-07-01 00:00:00,NaN
PM2.5,24933.0,NaN,NaN,NaN,67.450578,0.04,28.82,48.57,80.59,949.99,64.661449
PM10,18391.0,NaN,NaN,NaN,118.127103,0.01,56.255,95.68,149.745,1000.0,90.60511
NO,25949.0,NaN,NaN,NaN,17.57473,0.02,5.63,9.89,19.95,390.68,22.785846
NO2,25946.0,NaN,NaN,NaN,28.560659,0.01,11.75,21.69,37.62,362.21,24.474746
NOx,25346.0,NaN,NaN,NaN,32.309123,0.0,12.82,23.52,40.1275,467.63,31.646011
NH3,19203.0,NaN,NaN,NaN,23.483476,0.01,8.58,15.85,30.02,352.89,25.684275
CO,27472.0,NaN,NaN,NaN,2.248598,0.0,0.51,0.89,1.45,175.81,6.962884
SO2,25677.0,NaN,NaN,NaN,14.531977,0.01,5.67,9.16,15.22,193.86,18.133775



AQI bucket counts:


,Count
AQI_Bucket,
Moderate,8829
Satisfactory,8224
NaN,4681
Poor,2781
Very Poor,2337
Good,1341
Severe,1338
